In [17]:
import re
import numpy as np
import pandas as pd

# 2. Define the path to your file in Drive
# (Update this path to match where your dataset is located in your Drive)
file_path = '../data/raw/track1_merchants_master.csv'

# 3. Load dataset
merchant = pd.read_csv(file_path)

In [18]:
#column merchant id

# 1. standardization function
def std_id(value, prefix='MCH'):
    if pd.isnull(value):
        return None
    digits = re.sub(r'[^0-9]', '', str(value))
    return f"{prefix}{digits}" if digits else None

# 2. Apply standardization to clean merchant_id
merchant['merchant_id'] = merchant['merchant_id'].apply(std_id)

# 3. Categorize into 'Found Duplicate' and 'Unique'
# (keep=False marks ALL occurrences of duplicated IDs as True)
merchant['merchant_id_category'] = merchant['merchant_id'].duplicated(keep=False).map({
    True: 'Found Duplicate',
    False: 'Unique'
})

# 4. Display head 20 results
print(merchant[['merchant_id', 'merchant_id_category']].head(20))

   merchant_id merchant_id_category
0      MCH2849      Found Duplicate
1      MCH4314               Unique
2      MCH1986      Found Duplicate
3      MCH3899               Unique
4      MCH4859      Found Duplicate
5      MCH6848               Unique
6      MCH9933               Unique
7      MCH2637               Unique
8      MCH8708               Unique
9      MCH2430      Found Duplicate
10     MCH8556               Unique
11     MCH9997      Found Duplicate
12     MCH7003      Found Duplicate
13     MCH6945               Unique
14     MCH4323               Unique
15     MCH1405      Found Duplicate
16     MCH9459      Found Duplicate
17     MCH7645               Unique
18     MCH9620      Found Duplicate
19     MCH9815      Found Duplicate


In [19]:
#column merchant names

print("Missing merchant names:", merchant['merchant_name'].isnull().sum())
print("Unique merchant names:", merchant['merchant_name'].nunique())

Missing merchant names: 0
Unique merchant names: 5732


In [20]:

def clean_merchant_name(name):
    if pd.isnull(name) or str(name).strip().lower() in ['nan', 'none', '', 'na']:
        return 'Unknown Merchant'

    # Convert to string and strip surrounding whitespace
    text = str(name).strip()

    # Replace hyphens and underscores with spaces to normalize formatting
    text = re.sub(r'[-_]+', ' ', text)

    # Remove extra internal whitespace
    text = re.sub(r'\s+', ' ', text)

    # Standardize to Title Case (or use .capitalize() for strict sentence case)
    return text.title()

# Apply the function to the merchant name column
merchant['merchant_name'] = merchant['merchant_name'].apply(clean_merchant_name)

In [21]:
#column merchant_category


# 1. Sort and view unique categories before cleaning
print('Unique categories count:', merchant['merchant_category'].nunique())


# 2. Function to standardize and group similar merchant categories
def standardize_category(val):
  if pd.isnull(val):
    return 'Unknown'

  # Convert to lowercase and clean underscores/spaces
  val_clean = str(val).lower().replace(r'\_', '_').replace('_', ' ').strip()

  # Map variations to standard industry categories
  if val_clean in ['apparel', 'cloths', 'clothing', 'fashion', 'garments']:
    return 'Apparel'
  elif val_clean in ['books', 'book store', 'stationery', 'books stationery']:
    return 'Books & Stationery'
  elif val_clean in ['bus/taxi', 'transport', 'transportation', 'travel']:
    return 'Transport & Travel'
  elif val_clean in ['dept store', 'department store', 'department stores']:
    return 'Department Store'
  elif val_clean in ['food', 'eating place', 'restaurant', 'restaurants', 'food services']:
    return 'Food & Restaurant'
  elif val_clean in ['grocery', 'grocery stores', 'kirana', 'groceries', 'grocery store']:
    return 'Grocery'
  elif val_clean in ['hotel', 'hotels', 'hotel lodging', 'hospitality']:
    return 'Hospitality & Hotels'
  elif val_clean in ['medical', 'medical store', 'pharmacy', 'pharmacies', 'chemist']:
    return 'Healthcare & Pharmacy'
  elif val_clean in ['telecom', 'phone service', 'mobile recharge']:
    return 'Telecom'
  elif val_clean in ['retail']:
    return 'Retail'
  else:
    return 'Other / Misc'


# 3. Apply the standardization function
merchant['merchant_category'] = merchant['merchant_category'].apply(standardize_category)

# 4. Sort the DataFrame alphabetically by category name
merchant = merchant.sort_values(by='merchant_category').reset_index(drop=True)

# 5. Check the sorted and grouped category counts
print('\nStandardized Category Counts:')
print(merchant['merchant_category'].value_counts())

Unique categories count: 82

Standardized Category Counts:
merchant_category
Other / Misc             729
Apparel                  647
Telecom                  635
Healthcare & Pharmacy    622
Hospitality & Hotels     622
Grocery                  605
Food & Restaurant        597
Books & Stationery       593
Transport & Travel       519
Department Store         468
Retail                   173
Name: count, dtype: int64


In [22]:
#column MCC code

# 1. Clean MCC to strictly 4 digits
def clean_mcc(value):
  if pd.isnull(value):
    return None
  digits = re.sub(r'[^0-9]', '', str(value))
  return digits if len(digits) == 4 else None


merchant['clean_mcc'] = merchant['mcc'].apply(clean_mcc)

# 2. Build a mapping dictionary from merchant_category to the most common 4-digit MCC
valid_df = merchant.dropna(subset=['clean_mcc', 'merchant_category'])
cat_to_mcc = (
    valid_df.groupby('merchant_category')['clean_mcc']
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

# 3. Fill missing or invalid MCCs using the merchant_category mapping
merchant['clean_mcc'] = merchant['clean_mcc'].fillna(merchant['merchant_category'].map(cat_to_mcc))

# 4. Replace the old mcc column with the mapped/cleaned values
merchant['mcc'] = merchant['clean_mcc']
merchant = merchant.drop(columns=['clean_mcc'])

print(
    f'Remaining missing MCCs after mapping:'
    f" {merchant['mcc'].isnull().sum()}/{len(merchant)}"
)
print(merchant[['merchant_category', 'mcc']].head(15))

Remaining missing MCCs after mapping: 0/6210
   merchant_category   mcc
0            Apparel  5699
1            Apparel  5699
2            Apparel  5699
3            Apparel  5699
4            Apparel  5699
5            Apparel  5699
6            Apparel  5699
7            Apparel  5699
8            Apparel  5699
9            Apparel  5699
10           Apparel  5699
11           Apparel  5699
12           Apparel  5699
13           Apparel  5699
14           Apparel  5699


In [23]:
#coloumn business_type


# Clean and standardize business_type column
def clean_business_type(val):
  if pd.isnull(val):
    return 'Unknown'

  # Lowercase, fix underscores, hyphens, and extra spaces
  val_clean = (
      str(val)
      .lower()
      .replace(r'\_', '_')
      .replace('_', ' ')
      .replace('-', ' ')
      .strip()
  )

  # Group into standardized business types
  if 'private' in val_clean or 'pvt' in val_clean:
    return 'Private Limited'
  elif 'sole' in val_clean or 'proprietor' in val_clean:
    return 'Sole Proprietorship'
  elif 'partnership' in val_clean:
    return 'Partnership'
  elif 'individual' in val_clean:
    return 'Individual'
  else:
    return 'Other / Unknown'


# Apply transformation
merchant['business_type'] = merchant['business_type'].apply(clean_business_type)

# Check cleaned counts
print(merchant['business_type'].value_counts())

business_type
Individual             1596
Sole Proprietorship    1556
Private Limited        1547
Partnership            1511
Name: count, dtype: int64


In [24]:

# 1. Define city mapping dictionary for abbreviations and synonyms
city_map = {
    'hyd': 'Hyderabad',
    'hyderabad': 'Hyderabad',
    'jpr': 'Jaipur',
    'jaipur': 'Jaipur',
    'chennai': 'Chennai',
    'madras': 'Chennai',
    'ludhiana': 'Ludhiana',
    'ldh': 'Ludhiana',
    'jalandhar': 'Jalandhar',
    'jalandar': 'Jalandhar',
    'amritsar': 'Amritsar',
    'asr': 'Amritsar',
    'lucknow': 'Lucknow',
    'lko': 'Lucknow',
    'delhi': 'Delhi',
    'dilli': 'Delhi',
    'new delhi': 'Delhi',
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'kolkata': 'Kolkata',
    'calcutta': 'Kolkata',
    'pune': 'Pune',
    'poona': 'Pune',
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'blr': 'Bengaluru',
}


# 2. Cleaning function
def clean_city(val):
  if pd.isnull(val):
    return 'Unknown'
  val_clean = str(val).lower().strip()
  # Return mapped standard city or fallback to title case
  return city_map.get(val_clean, str(val).strip().title())


# 3. Apply transformation
merchant['city'] = merchant['city'].apply(clean_city)

# Filter out accidental header strings if any remain
merchant = merchant[merchant['city'] != 'City']

# Check clean counts
print(merchant['city'].value_counts())

city
Chennai      556
Bengaluru    540
Kolkata      536
Delhi        535
Pune         518
Jalandhar    514
Hyderabad    510
Lucknow      508
Amritsar     506
Jaipur       492
Ludhiana     479
Mumbai       431
Mumbay        85
Name: count, dtype: int64


In [25]:
#coloumn state

# Cleaning function for state
def clean_state(val):
  if pd.isnull(val):
    return 'Unknown'
  return str(val).strip().title()


# Apply transformation
merchant['state'] = merchant['state'].apply(clean_state)

# Filter out accidental header strings if any remain
merchant = merchant[merchant['state'] != 'State']

# Check clean counts
print(merchant['state'].value_counts())

state
Punjab           1499
Maharashtra      1034
Tamil Nadu        556
Karnataka         540
West Bengal       536
Delhi             535
Telangana         510
Uttar Pradesh     508
Rajasthan         492
Name: count, dtype: int64


In [26]:
#column onboarding date


# Cleaning function that returns a clean string date (avoids NaT)
def clean_date(val):
  if (
      pd.isnull(val)
      or str(val).strip() == ''
      or str(val).strip().lower() == 'onboarding_date'
  ):
    return np.nan
  val_str = str(val).strip()

  # Handle numeric UNIX timestamps
  if val_str.isdigit():
    try:
      return pd.to_datetime(int(val_str), unit='s').strftime('%Y-%m-%d')
    except Exception:
      pass

  # Parse string date formats
  try:
    return pd.to_datetime(val_str, errors='raise').strftime('%Y-%m-%d')
  except Exception:
    try:
      return pd.to_datetime(val_str, dayfirst=True, errors='raise').strftime(
          '%Y-%m-%d'
      )
    except Exception:
      return np.nan


# Apply transformation (keeps it as clean string format)
merchant['onboarding_date'] = merchant['onboarding_date'].apply(clean_date)

# Check results
print('Successfully parsed dates:', merchant['onboarding_date'].notnull().sum())
print(merchant['onboarding_date'].head(10))

/var/folders/5g/jklz980j60z7zg_2p9x50jqw0000gp/T/ipykernel_37666/2095979605.py:23: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val_str, errors='raise').strftime('%Y-%m-%d')
/var/folders/5g/jklz980j60z7zg_2p9x50jqw0000gp/T/ipykernel_37666/2095979605.py:23: UserWarning: Parsing dates in %d/%m/%Y %I:%M %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val_str, errors='raise').strftime('%Y-%m-%d')


Successfully parsed dates: 5711
0    2025-01-31
1    2023-05-29
2    2024-04-04
3    2024-07-16
4    2025-08-08
5    2023-10-18
6    2024-02-20
7    2023-01-30
8    2023-12-31
9    2024-08-18
Name: onboarding_date, dtype: object


In [27]:
# Create a separate status column
merchant['onboarding_date_status'] = merchant['onboarding_date'].apply(
    lambda x: 'missing' if pd.isnull(x) or str(x).strip().lower() in ['nan', 'none', 'null', ''] else 'present'
)

# Check distribution
print(merchant['onboarding_date_status'].value_counts())

onboarding_date_status
present    5711
missing     499
Name: count, dtype: int64


In [28]:
#column settlement account


# 1. Flag Settlement Account Data Quality Issues
def categorize_account(val):
    if pd.isnull(val):
        return 'MISSING'
    val_str = str(val).strip()
    if val_str.startswith('XXXX'):
        return 'MASKED'
    elif val_str.isdigit():
        return 'VALID_NUMERIC'
    elif val_str.isalnum():
        return 'ALPHANUMERIC_IBAN'
    return 'INVALID'

merchant['settlement_account_status'] = merchant['settlement_account'].apply(categorize_account)





In [29]:
# column merchant status

# FIX: preserve Suspended/Blocked/Hold as its own distinct category instead
# of collapsing it into Inactive — meaningfully different risk signal for
# a fraud-focused project. Also: missing status is now flagged as UNKNOWN,
# not fabricated via mode-imputation — we don't actually know if an
# unlabeled merchant is Active, Inactive, or Suspended.
def clean_status(val):
    if pd.isnull(val):
        return None
    val_str = str(val).strip().lower()
    if val_str in ['active', '1', 'true', 'y', 'enabled', 'live']:
        return 'Active'
    elif val_str in ['suspended', 's', 'hold', 'blocked']:
        return 'Suspended'
    elif val_str in ['inactive', 'i', 'closed', 'disabled', '0', 'false', 'n']:
        return 'Inactive'
    return 'Unknown'  # anything genuinely unrecognized, kept visible

merchant['merchant_status_clean'] = merchant['merchant_status'].apply(clean_status)

# Flag missing separately from "recognized but unusual" — don't fabricate
merchant['merchant_status_status'] = np.where(
    merchant['merchant_status'].isna(), 'MISSING', 'PROVIDED'
)

merchant['merchant_status'] = merchant['merchant_status_clean'].fillna('UNKNOWN')
merchant = merchant.drop(columns=['merchant_status_clean'])

print(merchant['merchant_status'].value_counts())
print(merchant['merchant_status_status'].value_counts())

merchant_status
Active       4076
Unknown       961
Inactive      759
Suspended     414
Name: count, dtype: int64
merchant_status_status
PROVIDED    6210
Name: count, dtype: int64


In [30]:
# column declared_avg_ticket_size

# FIX: add a status flag BEFORE imputing, so imputed values stay
# distinguishable from genuinely-reported ones — same pattern used
# consistently everywhere else in this project.
merchant['declared_avg_ticket_size'] = pd.to_numeric(
    merchant['declared_avg_ticket_size'].astype(str).str.replace(r'[^0-9.]', '', regex=True),
    errors='coerce'
)

merchant['ticket_size_status'] = np.where(
    merchant['declared_avg_ticket_size'].isna(), 'MISSING_OR_INVALID', 'PROVIDED'
)

# Impute only for downstream numeric convenience — the status column above
# is what tells you which values are real vs. filled.
merchant['declared_avg_ticket_size'] = merchant['declared_avg_ticket_size'].fillna(
    merchant.groupby('merchant_category')['declared_avg_ticket_size'].transform('median')
)
merchant['declared_avg_ticket_size'] = merchant['declared_avg_ticket_size'].fillna(
    merchant['declared_avg_ticket_size'].median()
)

print(merchant['ticket_size_status'].value_counts())

ticket_size_status
PROVIDED              5839
MISSING_OR_INVALID     371
Name: count, dtype: int64


In [31]:
merchant.to_csv('../data/cleaned/modified_merchant_master.csv', index=False)

In [32]:
merchant

,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state,onboarding_date,settlement_account,merchant_status,declared_avg_ticket_size,merchant_id_category,onboarding_date_status,settlement_account_status,merchant_status_status,ticket_size_status
0,MCH7236,Bhasin Llc,5699,Apparel,Partnership,Jalandhar,Punjab,2025-01-31,XXXX8980,Active,0.1228,Found Duplicate,present,MASKED,PROVIDED,PROVIDED
1,MCH2607,"Deep, Prasad And Dass",5699,Apparel,Private Limited,Hyderabad,Telangana,2023-05-29,4668596918,Inactive,1570.8200,Found Duplicate,present,VALID_NUMERIC,PROVIDED,PROVIDED
2,MCH8023,Char Gala,5699,Apparel,Individual,Chennai,Tamil Nadu,2024-04-04,PWFC1818550515264,Active,837.3500,Unique,present,ALPHANUMERIC_IBAN,PROVIDED,PROVIDED
3,MCH7258,"Om, Swaminathan And Ray",5699,Apparel,Private Limited,Jaipur,Rajasthan,2024-07-16,ARZZ7929473227025,Inactive,472.8900,Unique,present,ALPHANUMERIC_IBAN,PROVIDED,PROVIDED
4,MCH9180,Ramaswamyplc,5699,Apparel,Individual,Pune,Maharashtra,2025-08-08,NaN,Active,1761.2600,Found Duplicate,present,MISSING,PROVIDED,PROVIDED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,MCH3842,"Rao, Rajagopal And Ratti",4131,Transport & Travel,Private Limited,Jaipur,Rajasthan,2024-01-27,NaN,Active,444.1000,Found Duplicate,present,MISSING,PROVIDED,PROVIDED
6206,MCH5280,Mani Chada,4131,Transport & Travel,Sole Proprietorship,Chennai,Tamil Nadu,2025-03-18,XXXX6717,Active,1112.2400,Unique,present,MASKED,PROVIDED,PROVIDED
6207,MCH9444,Sachdeva Khatri,4131,Transport & Travel,Partnership,Jalandhar,Punjab,2024-01-13,2308250236,Active,345.2600,Found Duplicate,present,VALID_NUMERIC,PROVIDED,PROVIDED
6208,MCH7768,Sahota Walia,4131,Transport & Travel,Partnership,Delhi,Delhi,2025-06-19,NaN,Suspended,1097.5500,Unique,present,MISSING,PROVIDED,PROVIDED
